In [1]:
"""
====================================================================
ABLATION 4.1 (v2) -- LINEAR PROBE vs. FINE-TUNE-LAST-LAYER
Cross-Centre Generalization on Camelyon17-WILDS
====================================================================

STANDALONE SCRIPT -- does not import or touch downstream_classification_extended.py
or any of its cached files (features/checkpoints/results all live under their own
`/kaggle/working/finetune_ablation/` tree).

CHANGES FROM v1 (which was cancelled after 34588.8s, still not done):
  - DROPPED Variant C (fine-tune the FULL backbone). It was the single most
    expensive thing in the script (full backward pass through the entire
    ResNet/ViT/Swin backbone, 5 epochs, on the full ~365K-patch train set) and
    the least informative for what you're actually trying to show: full
    fine-tuning tends to wash out the exact signal (does reconstruction-
    pretraining specifically help?) that this ablation exists to probe.
    Variant B (frozen backbone, real gradient-trained head) already answers
    the reviewer's actual question -- "did you try letting it train at all,
    not just sklearn on frozen features?" -- for a fraction of the compute.
  - INCREMENTAL SAVING: v1 only wrote its CSV once, at the very end of the
    whole script. Cancelling it (as you did) lost every model's results, even
    ones that had already fully finished. v2 rewrites the CSV after EVERY
    variant (so at most one model x variant's worth of work is ever at risk),
    not just at the end.

WHAT THIS ANSWERS (Part-I completeness ablation, not a Part-II novelty claim):
Your entire evaluation so far is "freeze the reconstruction-pretrained backbone,
extract features once, fit a linear probe." That's the standard SSL evaluation
protocol, but a reviewer's first question is "did you even try fine-tuning, and
does the cross-centre gap change?" This script answers that, for each reconstruction
backbone (+ the ImageNet baseline for reference), with TWO variants:

    A. Linear probe        -- backbone fully frozen, sklearn LogisticRegression
                               on extracted features (identical protocol to what
                               you've already run elsewhere -- this is the anchor).
    B. Fine-tune last layer -- backbone fully frozen, but the classification head
                               is a trainable nn.Linear trained end-to-end with
                               gradient descent (not sklearn) for a few epochs.

Evaluated on the PRIMARY split only (centres 0,1,3,4 -> train, centre 2 -> test) to
keep this cheap -- a quick completeness check, not a second LOCO study. Every
AUROC also gets a patient-level bootstrap 95% CI, same as the main downstream
script, for consistency.

Kaggle notebook -- paste and run.
====================================================================
"""

import os
import gc
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score, confusion_matrix, matthews_corrcoef,
)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG -- entirely separate paths from downstream_classification_extended.py
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CHECKPOINT_DIR = "/kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints"

    WORK_DIR = "/kaggle/working/finetune_ablation"
    RESULTS_DIR = f"{WORK_DIR}/results"

    RECON_MODELS = {
        'recon_resnet18':  {'checkpoint': f"{CHECKPOINT_DIR}/v3_resnet18_best.pt",
                           'backbone': 'resnet18',  'input_size': 96},
        'recon_vit_small': {'checkpoint': f"{CHECKPOINT_DIR}/v3_vit_small_best.pt",
                           'backbone': 'vit_small', 'input_size': 224},
        'recon_swin_tiny': {'checkpoint': f"{CHECKPOINT_DIR}/v2_swin_tiny_best.pt",
                           'backbone': 'swin_tiny', 'input_size': 224},
    }
    # ImageNet reference row -- included so you can also answer "does fine-tuning
    # help a reconstruction-pretrained backbone MORE or LESS than an ImageNet one?"
    INCLUDE_IMAGENET_BASELINE = True

    TRAIN_CENTERS = [0, 1, 3, 4]
    TEST_CENTER = 2

    BATCH_SIZE = 64
    NUM_WORKERS = 2
    SEED = 42

    # -- variant B: fine-tune last layer only --
    HEAD_EPOCHS = 5
    HEAD_LR = 1e-3

    LR_MAX_ITER = 2000
    LR_C = 1.0

    BOOTSTRAP_N_RESAMPLES = 1000
    BOOTSTRAP_SEED = 7


os.makedirs(Config.RESULTS_DIR, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


# ====================================================================
# DATA
# ====================================================================
def load_full_dataset():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"OK loaded {len(full):,} patches")
    return full


class PatchDataset(Dataset):
    def __init__(self, hf_dataset, indices, input_size):
        self.ds = hf_dataset
        self.indices = indices
        self.transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        item = self.ds[int(idx)]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {'image': self.transform(img), 'label': int(item['label']),
                'center': int(item['center']), 'patient': int(item['patient'])}


# ====================================================================
# MODEL -- backbone (+ optional context transformer for recon models) + head
# ====================================================================
class ClassifierModel(nn.Module):
    """
    Wraps a backbone (reconstruction-pretrained or ImageNet) with a fresh binary
    classification head, and exposes `.set_mode('frozen' | 'head_only')` so the
    SAME model object can be used for both ablation variants without
    re-implementing the forward pass twice.
    """
    def __init__(self, backbone_name, checkpoint_path=None, input_size=96):
        super().__init__()
        self.backbone_name = backbone_name
        self.input_size = input_size
        self._build(backbone_name, checkpoint_path)
        self.head = nn.Linear(self.feature_dim, 1)

    def _build(self, backbone_name, checkpoint_path):
        import torchvision.models as tvm

        if backbone_name == 'imagenet_resnet18':
            net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.context = None
            return

        # ---- reconstruction-pretrained backbones: same architecture as training ----
        if backbone_name == 'resnet18':
            net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)
        elif backbone_name == 'vit_small':
            import timm
            self.backbone = timm.create_model('vit_small_patch16_224', pretrained=True,
                                              num_classes=0, img_size=self.input_size)
            self.feature_dim, self.pool = 384, None
        elif backbone_name == 'swin_tiny':
            import timm
            self.backbone = timm.create_model('swin_tiny_patch4_window7_224',
                                              pretrained=True, num_classes=0)
            self.feature_dim, self.pool = 768, None
        else:
            raise ValueError(f"unknown backbone {backbone_name}")

        self.proj = nn.Sequential(nn.Linear(self.feature_dim, self.feature_dim),
                                  nn.LayerNorm(self.feature_dim))
        self.context = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=self.feature_dim, nhead=8,
                                       dim_feedforward=min(4 * self.feature_dim, 2048),
                                       dropout=0.2, batch_first=True, norm_first=True),
            num_layers=4)
        self.slot_embed = nn.Parameter(torch.zeros(1, 9, self.feature_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.feature_dim))

        if checkpoint_path and os.path.exists(checkpoint_path):
            ck = torch.load(checkpoint_path, map_location='cpu')
            state = ck['model_state_dict']
            if list(state.keys())[0].startswith('module.'):
                state = {k.replace('module.', ''): v for k, v in state.items()}
            missing, unexpected = self.load_state_dict(state, strict=False)
            print(f"    loaded {checkpoint_path}  "
                  f"(missing={len(missing)}, unexpected={len(unexpected)} -- "
                  f"'head.*' missing is expected, it's freshly initialized)")

    def backbone_forward(self, x):
        """Single real patch, no jigsaw/context needed for classification --
        matches how features were extracted everywhere else in this project."""
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f

    def forward(self, x):
        f = self.backbone_forward(x)
        return self.head(f).squeeze(1)

    def set_mode(self, mode):
        """'frozen' (variant A: everything frozen, used only for feature extraction)
        or 'head_only' (variant B: freeze backbone, train head)."""
        for p in self.backbone.parameters():
            p.requires_grad = False
        if self.context is not None:
            for p in self.context.parameters():
                p.requires_grad = False
            for p in self.proj.parameters():
                p.requires_grad = False
        for p in self.head.parameters():
            p.requires_grad = (mode != 'frozen')


def build_classifier(name, cfg):
    if name == 'baseline_imagenet':
        return ClassifierModel('imagenet_resnet18', checkpoint_path=None, input_size=96)
    return ClassifierModel(cfg['backbone'], checkpoint_path=cfg['checkpoint'],
                          input_size=cfg['input_size'])


# ====================================================================
# METRICS
# ====================================================================
def compute_metrics(y_true, y_pred, y_prob):
    m = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'auroc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    m['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    m['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return m


def bootstrap_auroc_by_patient(y_true, y_prob, patient_ids,
                              n_boot=Config.BOOTSTRAP_N_RESAMPLES,
                              seed=Config.BOOTSTRAP_SEED):
    """Patient-cluster bootstrap 95% CI -- same rationale as the main downstream
    script: patches from one patient/slide aren't independent samples, so resample
    PATIENTS with replacement (keeping each one's patches together), not patches."""
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    idx_by_patient = {p: np.where(patient_ids == p)[0] for p in unique_patients}

    rng = np.random.RandomState(seed)
    boot_aurocs = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([idx_by_patient[p] for p in sampled])
        y_b, p_b = y_true[idx], y_prob[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_b, p_b))

    if not boot_aurocs:
        return {'auroc_ci_lo': float('nan'), 'auroc_ci_hi': float('nan'), 'n_boot_valid': 0}
    lo, hi = np.percentile(boot_aurocs, [2.5, 97.5])
    return {'auroc_ci_lo': float(lo), 'auroc_ci_hi': float(hi), 'n_boot_valid': len(boot_aurocs)}


# ====================================================================
# VARIANT A -- LINEAR PROBE (frozen backbone, sklearn probe -- the existing protocol)
# ====================================================================
@torch.no_grad()
def extract_features(model, hf_dataset, indices, input_size, device, tag='model'):
    model.eval().to(device)
    ds = PatchDataset(hf_dataset, indices, input_size)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    feats, labels, centers, patients = [], [], [], []
    for batch in tqdm(loader, desc=f"extract {tag}"):
        x = batch['image'].to(device, non_blocking=True)
        f = model.backbone_forward(x)
        feats.append(f.cpu().numpy())
        labels.extend(batch['label'].numpy())
        centers.extend(batch['center'].numpy())
        patients.extend(batch['patient'].numpy())
    return (np.vstack(feats), np.array(labels), np.array(centers), np.array(patients))


def eval_linear_probe(model, hf_dataset, train_idx, test_idx, device, tag):
    feats_tr, y_tr, _, _ = extract_features(model, hf_dataset, train_idx, model.input_size, device, tag=f"{tag}-train")
    feats_te, y_te, _, pat_te = extract_features(model, hf_dataset, test_idx, model.input_size, device, tag=f"{tag}-test")

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(feats_tr)
    X_te = scaler.transform(feats_te)
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, C=Config.LR_C,
                             class_weight='balanced', random_state=Config.SEED, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]

    m = compute_metrics(y_te, y_pred, y_prob)
    m.update(bootstrap_auroc_by_patient(y_te, y_prob, pat_te))
    return m


# ====================================================================
# VARIANT B -- GRADIENT-BASED TRAINING (head-only fine-tune)
# ====================================================================
def train_variant(model, hf_dataset, train_idx, device, mode, epochs, lr, tag):
    model.set_mode(mode)
    model.to(device)
    ds = PatchDataset(hf_dataset, train_idx, model.input_size)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(loader)))
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in tqdm(loader, desc=f"{tag} [{mode}] ep{epoch + 1}/{epochs}"):
            x = batch['image'].to(device, non_blocking=True)
            y = batch['label'].float().to(device, non_blocking=True)
            opt.zero_grad()
            logit = model(x)
            loss = criterion(logit, y)
            loss.backward()
            opt.step()
            sched.step()
            total_loss += loss.item() * x.size(0)
        print(f"  [{tag}][{mode}] epoch {epoch + 1}/{epochs}  loss={total_loss / len(ds):.4f}")
    model.eval()
    return model


@torch.no_grad()
def eval_classifier(model, hf_dataset, test_idx, device, tag):
    model.eval().to(device)
    ds = PatchDataset(hf_dataset, test_idx, model.input_size)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    probs, labels, patients = [], [], []
    for batch in tqdm(loader, desc=f"eval {tag}"):
        x = batch['image'].to(device, non_blocking=True)
        logit = model(x)
        probs.append(torch.sigmoid(logit).cpu().numpy())
        labels.extend(batch['label'].numpy())
        patients.extend(batch['patient'].numpy())
    probs = np.concatenate(probs)
    y_te = np.array(labels)
    pat_te = np.array(patients)
    preds = (probs >= 0.5).astype(int)
    m = compute_metrics(y_te, preds, probs)
    m.update(bootstrap_auroc_by_patient(y_te, probs, pat_te))
    return m


# ====================================================================
# MAIN
# ====================================================================
def main():
    set_seed(Config.SEED)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    csv_path = f"{Config.RESULTS_DIR}/finetune_ablation_{timestamp}.csv"

    print("=" * 70)
    print("ABLATION 4.1 (v2) -- LINEAR PROBE vs FINE-TUNE-LAST-LAYER")
    print("(variant C / full fine-tune DROPPED -- see docstring for why)")
    print("=" * 70)
    print(f"device: {device}\n")

    print("Loading Camelyon17-WILDS...")
    hf_dataset = load_full_dataset()
    n = len(hf_dataset)
    all_idx = np.arange(n)

    print("\nBuilding primary-split indices by CENTRE column (cheap, no image decode)...")
    centers_all = np.asarray(hf_dataset['center'], dtype=np.int64)
    train_idx = all_idx[np.isin(centers_all, Config.TRAIN_CENTERS)]
    test_idx = all_idx[centers_all == Config.TEST_CENTER]
    print(f"  train (centres {Config.TRAIN_CENTERS}): {len(train_idx):,} patches")
    print(f"  test  (centre {Config.TEST_CENTER}):        {len(test_idx):,} patches")

    models_to_run = dict(Config.RECON_MODELS)
    if Config.INCLUDE_IMAGENET_BASELINE:
        models_to_run['baseline_imagenet'] = {'backbone': 'imagenet_resnet18',
                                              'checkpoint': None, 'input_size': 96}

    rows = []

    def save_progress():
        """Rewrites the CSV after every variant finishes -- at most one model x
        variant's worth of work is ever at risk if the notebook gets stopped,
        instead of losing the entire run like v1 did."""
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f"  [checkpoint] {len(rows)} rows saved -> {csv_path}")

    for name, cfg in models_to_run.items():
        print(f"\n{'='*70}\nMODEL: {name}\n{'='*70}")

        # ---- variant A: linear probe (frozen, sklearn) ----
        print(f"\n[{name}] variant A -- linear probe (frozen)")
        model = build_classifier(name, cfg)
        model.set_mode('frozen')
        m_a = eval_linear_probe(model, hf_dataset, train_idx, test_idx, device, tag=name)
        m_a.update({'model': name, 'variant': 'A_linear_probe',
                   'n_train': len(train_idx), 'n_test': len(test_idx)})
        rows.append(m_a)
        print(f"  AUROC={m_a['auroc']:.4f}  [{m_a['auroc_ci_lo']:.4f}, {m_a['auroc_ci_hi']:.4f}]  "
              f"BalAcc={m_a['balanced_accuracy']:.4f}")
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        save_progress()

        # ---- variant B: fine-tune last layer (head trained via SGD, backbone frozen) ----
        print(f"\n[{name}] variant B -- fine-tune last layer only")
        model = build_classifier(name, cfg)
        model = train_variant(model, hf_dataset, train_idx, device, mode='head_only',
                             epochs=Config.HEAD_EPOCHS, lr=Config.HEAD_LR, tag=name)
        m_b = eval_classifier(model, hf_dataset, test_idx, device, tag=name)
        m_b.update({'model': name, 'variant': 'B_finetune_head',
                   'n_train': len(train_idx), 'n_test': len(test_idx)})
        rows.append(m_b)
        print(f"  AUROC={m_b['auroc']:.4f}  [{m_b['auroc_ci_lo']:.4f}, {m_b['auroc_ci_hi']:.4f}]  "
              f"BalAcc={m_b['balanced_accuracy']:.4f}")
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        save_progress()

    df = pd.DataFrame(rows)
    print(f"\nOK final results -> {csv_path}")

    print("\n" + "=" * 70)
    print("SUMMARY -- AUROC by model x variant")
    print("=" * 70)
    pivot = df.pivot_table(index='model', columns='variant', values='auroc')
    print(pivot.to_string())

    fig, ax = plt.subplots(figsize=(9, 5))
    pivot.plot(kind='bar', ax=ax)
    ax.set_ylabel('AUROC')
    ax.set_ylim(0.4, 1.0)
    ax.set_title('Linear probe vs fine-tune-head, per model (Centre 2 held out)')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20)
    plt.tight_layout()
    fig_path = f"{Config.RESULTS_DIR}/finetune_ablation_{timestamp}.png"
    plt.savefig(fig_path, dpi=150)
    plt.close(fig)
    print(f"OK {fig_path}")

    print("\n" + "=" * 70)
    print("DONE")
    print(f"   CSV    : {csv_path}")
    print(f"   Figure : {fig_path}")
    print("   NOTE: auroc_ci_lo/hi are patient-level bootstrap 95% CIs, same as the")
    print("         main downstream script -- not plain patch-level CIs.")
    print("   NOTE: this CSV was also rewritten after every single variant during")
    print("         the run -- if you stopped this notebook early, re-open the most")
    print("         recent finetune_ablation_*.csv in RESULTS_DIR; it holds every")
    print("         model x variant that finished before the stop.")
    print("=" * 70)
    return df


# ====================================================================
# RUN
# ====================================================================
df = main()

ABLATION 4.1 (v2) -- LINEAR PROBE vs FINE-TUNE-LAST-LAYER
(variant C / full fine-tune DROPPED -- see docstring for why)
device: cuda

Loading Camelyon17-WILDS...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ccce1d96840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ccce1d96840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  [baseline_imagenet][head_only] epoch 5/5  loss=0.2326


eval baseline_imagenet:   0%|          | 0/1329 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ccce1d96840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ccce1d96840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ccce1d96840>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

Traceback (most recent call last):
      File "/usr/local/lib/python3.12/d

  AUROC=0.8806  [0.8113, 0.9081]  BalAcc=0.7830
  [checkpoint] 8 rows saved -> /kaggle/working/finetune_ablation/results/finetune_ablation_20260925_065335.csv

OK final results -> /kaggle/working/finetune_ablation/results/finetune_ablation_20260925_065335.csv

SUMMARY -- AUROC by model x variant
variant            A_linear_probe  B_finetune_head
model                                             
baseline_imagenet        0.947297         0.880647
recon_resnet18           0.884784         0.779091
recon_swin_tiny          0.961428         0.954871
recon_vit_small          0.960492         0.962139
OK /kaggle/working/finetune_ablation/results/finetune_ablation_20260925_065335.png

DONE
   CSV    : /kaggle/working/finetune_ablation/results/finetune_ablation_20260925_065335.csv
   Figure : /kaggle/working/finetune_ablation/results/finetune_ablation_20260925_065335.png
   NOTE: auroc_ci_lo/hi are patient-level bootstrap 95% CIs, same as the
         main downstream script -- not plain patch-